In [1]:
"""
================================================================================
MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION  (v3.1 — bug-fix release)
With SOTA Comparisons, Deep Error Analysis, Ablations, Calibrated Cascade,
Seeded Reproducibility, and SHAP Explanations
================================================================================

CHANGES vs v3.0:

  F1  SHAP array-ambiguity crash fixed: run_shap_analysis() now handles both
      the list form and the 3-D array form returned by different shap
      versions, and coerces importance values to plain floats before sorting.

  F2  Ablations were silently no-ops: MemoryOptimizedDataPreprocessor,
      AdvancedImbalanceHandler, and EnhancedTwoStepFramework now take
      `ablation_mode` as an instance attribute instead of reading the
      module-level CONFIG. All four ablation flags now actually change
      behaviour.

  F3  CIC-IDS2017 and CSE-CICIDS2018 now load every per-day CSV/parquet
      file and concatenate, instead of picking only the single largest
      file. Both datasets were missing from the prior summary table because
      either their Kaggle paths differed or only one day's file was used.

  F4  Dataset-specific label handling added for CIC-IDS2017 and
      CSE-CICIDS2018 (strip whitespace on string labels, identify the label
      column explicitly).

  F5  main() now prints a present/missing diagnostic for every dataset,
      lists the contents of /kaggle/input when a path is missing, and
      writes failed_runs.csv with full tracebacks, so silently-skipped
      datasets cannot happen again.

Prior changes retained (R1–R12 from v3.0) are unchanged.

Author: Enhanced Research Implementation
Date: 2026-09-15
================================================================================
"""

# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score
)
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_classif, SelectKBest
)

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, callbacks
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False

import shap
import gc
import os
import time
import json
import logging
import traceback
from tqdm import tqdm
from datetime import datetime
import psutil
import warnings

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"✅ SHAP version: {shap.__version__}")

# ============================================================================
# GLOBAL SEEDED RNG
# ============================================================================

def make_rng(seed: int = 42) -> np.random.RandomState:
    """Single source of randomness for the whole pipeline."""
    return np.random.RandomState(seed)

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'sampling_rate': 0.15,
    'n_features': 30,
    'model_type': 'random_forest',
    'use_smote': True,
    'smote_type': 'BorderlineSMOTE',
    'leakage_mi_threshold': 0.95,
    'leakage_threshold': 0.9,          # kept for backwards compat in old reports
    'explainer_type': 'tree',
    'test_size': 0.2,
    'random_seed': 42,
    'n_folds': 3,
    'shap_samples': 100,
    'output_dir': '/kaggle/working/results/',
    'deep_learning_epochs': 20,
    'deep_learning_batch_size': 64,
    'max_rows': 500000,
    'chunk_size': 100000,
    'memory_threshold': 0.85,
    'max_categories_for_ohe': 50,

    # Repeated runs so the paper can report mean ± std.
    'repeats': 4,
    'repeat_seeds': [42, 43, 44, 45],

    # Ablation switches. Valid: None | 'no_leakage' | 'no_feature_selection'
    #                          | 'no_smote' | 'no_cascade'.
    'ablation_mode': None,

    # Actually run SHAP.
    'run_shap': True,
}

DATASETS = {
    'NSL-KDD': {
        'path': '/kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert',
        'file_patterns': ['KDD', 'kdd', 'nsl-kdd', 'NSL-KDD', '.csv', '.data'],
        'label_col': 'Label',
        'description': 'Benchmark dataset',
        'skip_files': ['.jpg', '.png', '.jpeg'],
        'max_rows': 200000
    },
    'UNSW-NB15': {
        'path': '/kaggle/input/datasets/likkisamarthreddy/unsw-nb15',
        'file_patterns': ['UNSW', 'unsw', 'UNSW-NB15', '.csv'],
        'label_col': 'Label',
        'description': 'Modern attack coverage',
        'max_rows': 300000
    },
    'CIC-IDS2017': {
        'path': '/kaggle/input/datasets/bertvankeulen/cicids-2017',
        'file_patterns': ['CIC-IDS2017', 'cicids2017', '.csv'],
        'label_col': 'Label',
        'description': 'Primary dataset with extreme class imbalance',
        'max_rows': 300000
    },
    'CSE-CICIDS2018': {
        'path': '/kaggle/input/datasets/shrey213/cicids2018',
        'file_patterns': ['CSE-CICIDS2018', 'csecicids2018', '.parquet', '.csv'],
        'label_col': 'Label',
        'description': 'Large-scale validation',
        'max_rows': 300000
    },
    'CIC-ToN-IoT': {
        'path': '/kaggle/input/datasets/wahidulislambayazid/cic-ton-iot',
        'file_patterns': ['ToN-IoT', 'toniot', 'CIC-ToN-IoT', '.parquet', '.csv'],
        'label_col': 'Label',
        'description': 'IoT-specific testing',
        'max_rows': 300000
    }
}

# ============================================================================
# MEMORY UTILITY FUNCTIONS
# ============================================================================

def get_memory_usage():
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024 / 1024

def check_memory(threshold=0.85):
    memory_gb = get_memory_usage()
    total_memory = psutil.virtual_memory().total / 1024 / 1024 / 1024
    usage_ratio = memory_gb / total_memory
    return usage_ratio < threshold, usage_ratio

def optimize_dataframe_dtypes(df):
    if not df.columns.is_unique:
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            dup_indices = cols[cols == dup].index
            for i, idx in enumerate(dup_indices):
                if i > 0:
                    cols.iloc[idx] = f"{dup}_{i}"
        df.columns = cols

    for col in df.columns:
        if not isinstance(df[col], pd.Series):
            continue
        col_type = df[col].dtype
        if col_type != 'object':
            try:
                if col_type.kind in ['i', 'u']:
                    c_min, c_max = df[col].min(), df[col].max()
                    if c_min >= 0:
                        if c_max < 255: df[col] = df[col].astype('uint8')
                        elif c_max < 65535: df[col] = df[col].astype('uint16')
                        elif c_max < 4294967295: df[col] = df[col].astype('uint32')
                        else: df[col] = df[col].astype('uint64')
                    else:
                        if c_min > -128 and c_max < 127: df[col] = df[col].astype('int8')
                        elif c_min > -32768 and c_max < 32767: df[col] = df[col].astype('int16')
                        elif c_min > -2147483648 and c_max < 2147483647: df[col] = df[col].astype('int32')
                        else: df[col] = df[col].astype('int64')
                elif col_type.kind == 'f':
                    df[col] = df[col].astype('float32')
            except Exception:
                pass
    return df

def cleanup_memory():
    gc.collect()
    if TF_AVAILABLE:
        tf.keras.backend.clear_session()
    print(f"   Memory after cleanup: {get_memory_usage():.2f} GB")

# ============================================================================
# STRATIFIED SUBSAMPLE HELPER
# ============================================================================

def stratified_subsample(X, y, n_samples, seed=42):
    """
    Stratified subsample of (X, y) to ~n_samples rows.
    Preserves class proportions so rare-attack classes are not accidentally
    dropped from the evaluation set.
    """
    if n_samples is None or n_samples >= len(X):
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=seed)
    idx, _ = next(sss.split(X, y))
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    return X[idx], y[idx]

# ============================================================================
# DATA PREPROCESSOR
# ============================================================================

class MemoryOptimizedDataPreprocessor:
    def __init__(self, dataset_name, leakage_threshold=0.9, seed=42,
                 ablation_mode=None):
        self.dataset_name = dataset_name
        self.leakage_threshold = leakage_threshold
        self.leaky_features = []
        self.kept_features = []
        self.label_encoder = LabelEncoder()
        self.preprocessing_time = 0
        self.original_columns = []
        self.label_col = 'Label'
        self.freq_encodings = {}
        self.rng = make_rng(seed)
        # CHANGED (F2): ablation mode now lives on the instance, not on the
        # module-level CONFIG, so each run actually applies its own ablation.
        self.ablation_mode = ablation_mode

    # ---------------------------------------------------------------------
    def load_data_from_kaggle(self, dataset_config):
        """
        CHANGED (F3, F5): multi-file concat for the per-day datasets, plus
        loud diagnostics when a path is missing so silently-skipped datasets
        cannot happen.
        """
        print(f"📂 Loading {self.dataset_name}...")
        start_time = time.time()
        dataset_path = dataset_config['path']
        skip_files = dataset_config.get('skip_files', [])
        max_rows = dataset_config.get('max_rows', CONFIG['max_rows'])

        if not os.path.exists(dataset_path):
            print(f"   ❌ Path not found: {dataset_path}")
            for guess in ['/kaggle/input', '/kaggle/input/datasets']:
                if os.path.exists(guess):
                    print(f"   ℹ️ Contents of {guess}:")
                    try:
                        for entry in sorted(os.listdir(guess))[:25]:
                            print(f"      - {entry}")
                    except Exception:
                        pass
            raise FileNotFoundError(
                f"{self.dataset_name} path not found: {dataset_path}. "
                f"Update DATASETS['{self.dataset_name}']['path'] to match the "
                f"actual Kaggle slug."
            )

        all_files = []
        for root, dirs, files in os.walk(dataset_path):
            for file in files:
                all_files.append(os.path.join(root, file))

        filtered_files = [f for f in all_files
                          if not any(s in f.lower() for s in skip_files)]
        if not filtered_files:
            raise FileNotFoundError(f"No suitable files found under {dataset_path}")

        print(f"   Found {len(filtered_files)} files under {dataset_path}")

        # ---------------------------------------------------------------
        # UNSW-NB15 specific selection (unchanged)
        # ---------------------------------------------------------------
        if self.dataset_name == 'UNSW-NB15':
            import re
            pattern = re.compile(r'unsw-nb15', re.IGNORECASE)
            csv_files = [f for f in filtered_files
                         if f.lower().endswith('.csv') and pattern.search(f)]
            if not csv_files:
                raise FileNotFoundError("No UNSW-NB15 CSV files found")
            chosen_file = None
            for f in csv_files:
                try:
                    sample = pd.read_csv(f, nrows=0, encoding='latin-1')
                    cols = [c.strip().lower().replace(' ', '_') for c in sample.columns]
                    if ('attack_cat' in cols or 'label' in cols) and len(cols) < 100:
                        chosen_file = f
                        break
                except Exception:
                    continue
            if chosen_file is None:
                chosen_file = max(csv_files, key=lambda x: os.path.getsize(x))
                print(f"   ⚠️ No standard UNSW-NB15 file, using largest: "
                      f"{os.path.basename(chosen_file)}")
            else:
                print(f"   Selected standard file: {os.path.basename(chosen_file)}")

            df = self._load_file_memory_optimized(chosen_file, max_rows)
            if df is None or len(df) == 0:
                raise RuntimeError("UNSW-NB15 load returned empty dataframe")
            df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
            df.columns = df.columns.str.replace('[^a-zA-Z0-9_]', '', regex=True)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = optimize_dataframe_dtypes(df)
            df = self._dataset_specific_preprocessing(df)
            self.original_columns = df.columns.tolist()
            print(f"   ✅ Loaded {len(df):,} rows, {len(df.columns)} cols "
                  f"in {time.time()-start_time:.2f}s")
            return df

        # ---------------------------------------------------------------
        # CHANGED (F3): multi-file loading for CIC-IDS2017 / CSE-CICIDS2018.
        # Both ship as one CSV/parquet per capture day. Picking only the
        # largest file (v3.0 behaviour) discarded most attack classes and in
        # practice also caused the dataset to appear "missing" from results.
        # ---------------------------------------------------------------
        multi_file_datasets = ('CIC-IDS2017', 'CSE-CICIDS2018')
        if self.dataset_name in multi_file_datasets:
            patterns = dataset_config.get('file_patterns', [])
            pattern_files = [f for f in filtered_files
                             if any(p.lower() in f.lower() for p in patterns)]
            data_files = pattern_files or [
                f for f in filtered_files
                if f.lower().endswith(('.csv', '.parquet'))
            ]
            data_files = sorted(data_files)
            print(f"   Multi-file mode: concatenating {len(data_files)} files")
            if not data_files:
                raise FileNotFoundError(
                    f"No CSV/parquet files found under {dataset_path}"
                )

            dfs = []
            rows_so_far = 0
            for f in data_files:
                try:
                    d = self._load_file_memory_optimized(f, None)
                    if d is None or len(d) == 0:
                        continue
                    dfs.append(d)
                    rows_so_far += len(d)
                    print(f"     + {os.path.basename(f)}: {len(d):,} rows "
                          f"(total {rows_so_far:,})")
                    # Early exit if we already have plenty; keeps memory bounded.
                    if max_rows and rows_so_far >= max_rows * 2:
                        print(f"     (stopping early — already have {rows_so_far:,})")
                        break
                except Exception as e:
                    print(f"     ⚠️ Skipped {os.path.basename(f)}: {e}")
                    continue

            if not dfs:
                raise RuntimeError(
                    f"No readable files under {dataset_path} — all loads failed"
                )

            df = pd.concat(dfs, ignore_index=True)
            del dfs
            gc.collect()

            if max_rows and len(df) > max_rows:
                print(f"   ⚠️ Combined {len(df):,} rows, subsampling to {max_rows:,}")
                df = df.sample(n=max_rows, random_state=CONFIG['random_seed'])
        else:
            selected_file = self._select_best_file(
                filtered_files, dataset_config.get('file_patterns', [])
            )
            if selected_file is None:
                raise FileNotFoundError(f"No readable file under {dataset_path}")
            file_size_mb = os.path.getsize(selected_file) / (1024 * 1024)
            print(f"   Selected: {os.path.basename(selected_file)} "
                  f"({file_size_mb:.2f} MB)")
            df = self._load_file_memory_optimized(selected_file, max_rows)

        # ---------------------------------------------------------------
        # Common column cleanup
        # ---------------------------------------------------------------
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
        df.columns = df.columns.str.replace('[^a-zA-Z0-9_]', '', regex=True)
        df = df.replace([np.inf, -np.inf], np.nan)
        df = optimize_dataframe_dtypes(df)
        df = self._dataset_specific_preprocessing(df)
        self.original_columns = df.columns.tolist()
        print(f"   ✅ Loaded {len(df):,} rows, {len(df.columns)} cols "
              f"in {time.time()-start_time:.2f}s")
        return df

    def _load_file_memory_optimized(self, file_path, max_rows):
        try:
            if file_path.endswith('.parquet'):
                try:
                    df = pd.read_parquet(file_path, engine='pyarrow')
                except Exception:
                    df = pd.read_parquet(file_path)
            elif file_path.endswith('.csv'):
                file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
                if file_size_mb > 500:
                    print(f"   ⚠️ Large CSV ({file_size_mb:.1f} MB), chunked read...")
                    chunks, n_chunks = [], 0
                    for encoding in ['utf-8', 'latin-1']:
                        try:
                            for chunk in pd.read_csv(
                                file_path, chunksize=CONFIG['chunk_size'],
                                low_memory=False, encoding=encoding
                            ):
                                chunks.append(chunk)
                                n_chunks += 1
                                if (max_rows is not None
                                        and n_chunks * CONFIG['chunk_size'] >= max_rows):
                                    break
                            df = pd.concat(chunks, ignore_index=True)
                            break
                        except (UnicodeDecodeError, pd.errors.ParserError):
                            if encoding == 'latin-1':
                                raise
                            continue
                else:
                    df = None
                    for encoding in ['utf-8', 'latin-1']:
                        try:
                            df = pd.read_csv(file_path, low_memory=False,
                                             encoding=encoding)
                            break
                        except (UnicodeDecodeError, pd.errors.ParserError):
                            if encoding == 'latin-1':
                                raise
                            continue
                    if df is None:
                        raise RuntimeError(f"Could not read {file_path}")
            elif file_path.endswith(('.txt', '.data')):
                try:
                    df = pd.read_csv(file_path, low_memory=False)
                except Exception:
                    df = pd.read_csv(file_path, header=None, low_memory=False)
                    if len(df.columns) == 42:
                        df.columns = ([f'feature_{i}'
                                       for i in range(len(df.columns) - 1)]
                                      + ['Label'])
            else:
                df = pd.read_csv(file_path, low_memory=False)

            if max_rows is not None and len(df) > max_rows:
                print(f"   ⚠️ Dataset has {len(df):,} rows, sampling to {max_rows:,}")
                df = df.sample(n=max_rows, random_state=CONFIG['random_seed'])
            return df
        except Exception as e:
            print(f"   ❌ Error loading file {os.path.basename(file_path)}: {e}")
            return None

    def _select_best_file(self, files, patterns):
        matching = [f for f in files
                    if any(p.lower() in f.lower() for p in patterns)]
        if matching:
            return max(matching, key=lambda x: os.path.getsize(x))
        data_files = [f for f in files
                      if f.endswith(('.csv', '.parquet', '.feather', '.data', '.txt'))]
        if data_files:
            return max(data_files, key=lambda x: os.path.getsize(x))
        return files[0] if files else None

    # ---------------------------------------------------------------------
    def _dataset_specific_preprocessing(self, df):
        if self.dataset_name == 'UNSW-NB15':
            label_candidates = ['attack_cat', 'label', 'class', 'Label', 'Attack_cat']
            self.label_col = next((c for c in label_candidates if c in df.columns), None)
            if self.label_col is None:
                self.label_col = self.identify_label_column(df)
            n_unique = df[self.label_col].nunique(dropna=True)
            if n_unique < 2:
                raise ValueError(
                    f"UNSW-NB15 label '{self.label_col}' has only "
                    f"{n_unique} unique value(s)."
                )
        elif self.dataset_name == 'NSL-KDD':
            for c in ['label', 'class', 'Label']:
                if c in df.columns:
                    self.label_col = c
                    break
        elif self.dataset_name == 'CIC-ToN-IoT':
            for col in ['label', 'Label', 'attack', 'Attack', 'class', 'Class']:
                if col in df.columns:
                    self.label_col = col
                    break
            for col in df.columns:
                if df[col].dtype == 'object':
                    continue
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    pass
        # CHANGED (F4): explicit label handling for the two CIC datasets.
        elif self.dataset_name in ('CIC-IDS2017', 'CSE-CICIDS2018'):
            label_candidates = ['label', 'Label', 'attack', 'Attack', 'class', 'Class']
            self.label_col = next(
                (c for c in label_candidates if c in df.columns), None
            )
            if self.label_col is None:
                self.label_col = self.identify_label_column(df)
            # Raw CIC files often have leading/trailing whitespace in labels
            # and duplicate column names (e.g. ' Fwd Header Length.1').
            if df[self.label_col].dtype == 'object':
                df[self.label_col] = df[self.label_col].astype(str).str.strip()
            n_unique = df[self.label_col].nunique(dropna=True)
            print(f"   Label column: '{self.label_col}', "
                  f"{n_unique} unique classes")
            if n_unique < 2:
                raise ValueError(
                    f"{self.dataset_name} label '{self.label_col}' has only "
                    f"{n_unique} unique value(s) after preprocessing."
                )
        return df

    def identify_label_column(self, df):
        pats = ['label', 'class', 'attack_cat', 'attack_type', 'category',
                'normal', 'Label', 'Class', 'Attack_Cat']
        for col in df.columns:
            if any(p.lower() in col.lower() for p in pats):
                return col
        for col in df.columns:
            try:
                if (df[col].nunique() <= 5
                        and df[col].dtype in ['int64', 'float64', 'int32']):
                    return col
            except Exception:
                pass
        return df.columns[-1]

    def _encode_categorical_features(self, X):
        obj_cols = X.select_dtypes(include=['object']).columns.tolist()
        if not obj_cols:
            return X
        X_enc = X.copy()
        for col in obj_cols:
            n_unique = X_enc[col].nunique(dropna=False)
            if n_unique <= CONFIG['max_categories_for_ohe']:
                dummies = pd.get_dummies(X_enc[col], prefix=col, drop_first=True)
                X_enc = pd.concat([X_enc.drop(columns=[col]), dummies], axis=1)
            else:
                freq_map = X_enc[col].value_counts().to_dict()
                self.freq_encodings[col] = freq_map
                X_enc[col] = X_enc[col].map(freq_map).astype('float32')
        return X_enc

    # ---------------------------------------------------------------------
    def identify_leaky_features(self, df, label_col='Label'):
        """Leakage screening via mutual information (nominal-label safe)."""
        print("🔍 Identifying leaky features (mutual information)…")
        start_time = time.time()

        if label_col not in df.columns:
            label_col = self.identify_label_column(df)
        self.label_col = label_col
        print(f"   Using label column: '{label_col}'")

        X = df.drop(columns=[label_col])
        y = df[label_col].astype(str)
        X = self._encode_categorical_features(X)

        n_unique_labels = y.nunique(dropna=True)
        if n_unique_labels < 2:
            raise ValueError(
                f"Label column '{label_col}' for '{self.dataset_name}' "
                f"has only {n_unique_labels} unique value(s)."
            )
        y_encoded = pd.factorize(y)[0]

        for col in X.columns:
            try:
                if X[col].dtype == 'object':
                    X[col] = pd.to_numeric(X[col], errors='coerce')
            except Exception:
                pass
        X = X.dropna(axis=1, how='all')
        X = X.fillna(0)

        sub_n = min(10000, len(X))
        if len(X) > sub_n:
            idx = self.rng.choice(len(X), sub_n, replace=False)
            X_sub = X.iloc[idx]
            y_sub = y_encoded[idx]
        else:
            X_sub = X
            y_sub = y_encoded

        mi_scores = {}
        try:
            valid_cols = [c for c in X_sub.columns if X_sub[c].nunique() > 1]
            if valid_cols:
                mi = mutual_info_classif(
                    X_sub[valid_cols].values.astype(float),
                    y_sub,
                    discrete_features=False,
                    random_state=self.rng.randint(0, 2**31 - 1)
                )
                mi_scores = dict(zip(valid_cols, mi))
        except Exception as e:
            print(f"   ⚠️ MI failed ({e}); falling back to no MI-based drops")
            mi_scores = {}

        if mi_scores:
            mi_max = max(mi_scores.values()) or 1.0
            mi_norm = {k: v / mi_max for k, v in mi_scores.items()}
        else:
            mi_norm = {}

        self.leaky_features = [c for c, v in mi_norm.items()
                               if v >= CONFIG['leakage_mi_threshold']]

        leak_patterns = ['time', 'date', 'timestamp', 'duration', 'flow_duration',
                         'id', 'index', 'seq', 'num', 'flow_id']
        for col in X.columns:
            if any(p in col.lower() for p in leak_patterns):
                if col not in self.leaky_features:
                    try:
                        if X[col].nunique() == len(X):
                            self.leaky_features.append(col)
                    except Exception:
                        pass

        self.kept_features = [c for c in X.columns if c not in self.leaky_features]
        print(f"   ✅ {len(self.leaky_features)} leaky features flagged, "
              f"{len(self.kept_features)} kept, in {time.time()-start_time:.2f}s")
        return X[self.kept_features], y, label_col

    # ---------------------------------------------------------------------
    def preprocess(self, df, label_col=None, test_mode=False):
        print("🔧 Preprocessing data...")
        start_time = time.time()
        df = df.copy().dropna(how='all').replace([np.inf, -np.inf], np.nan)
        if label_col is None or label_col not in df.columns:
            label_col = self.identify_label_column(df)

        # CHANGED (F2): read ablation from instance, not module CONFIG.
        if self.ablation_mode == 'no_leakage':
            print("   ⚠️ ABLATION: leakage screening disabled")
            X = df.drop(columns=[label_col])
            X = self._encode_categorical_features(X)
            y_str = df[label_col].astype(str)
        else:
            X, y_str, label_col = self.identify_leaky_features(df, label_col)

        normal_mask = y_str.str.lower().isin(['normal', 'benign'])
        if normal_mask.any():
            normal_class = y_str[normal_mask].iloc[0]
            all_classes = [normal_class] + sorted(
                [c for c in y_str.unique() if c != normal_class]
            )
            class_to_idx = {cls: i for i, cls in enumerate(all_classes)}
            y_encoded = y_str.map(class_to_idx).values
        else:
            print("   ⚠️ No 'normal'/'benign' class found — falling back to LabelEncoder.")
            if test_mode:
                y_encoded = self.label_encoder.transform(y_str)
            else:
                y_encoded = self.label_encoder.fit_transform(y_str)

        if len(X) == 0:
            return pd.DataFrame(), pd.Series()

        try:
            selector = VarianceThreshold(threshold=0.01)
            X_selected = selector.fit_transform(X)
            X_columns = X.columns[selector.get_support()].tolist()
        except Exception:
            X_selected = X.values
            X_columns = X.columns.tolist()

        X_final = pd.DataFrame(X_selected, columns=X_columns)
        for col in X_final.columns:
            try:
                X_final[col] = pd.to_numeric(X_final[col], errors='coerce')
            except Exception:
                pass
        X_final = X_final.dropna(axis=1, how='all').fillna(0)
        X_final = optimize_dataframe_dtypes(X_final)
        self.preprocessing_time = time.time() - start_time
        print(f"   ✅ Preprocessed: {len(X_final):,} rows, "
              f"{len(X_final.columns)} features in {self.preprocessing_time:.2f}s")
        del df
        cleanup_memory()
        return X_final, pd.Series(y_encoded, name='Label')

    def get_leakage_report(self):
        return {
            'dataset': self.dataset_name,
            'leaky_features': self.leaky_features,
            'leaky_count': len(self.leaky_features),
            'kept_count': len(self.kept_features),
            'original_columns': self.original_columns[:10],
            'preprocessing_time': self.preprocessing_time,
            'label_col': self.label_col,
            'method': 'mutual_info_classif',
        }

# ============================================================================
# ADVANCED IMBALANCE HANDLING
# ============================================================================

class AdvancedImbalanceHandler:
    def __init__(self, technique='SMOTE', random_seed=42, ablation_mode=None):
        self.technique = technique
        self.random_seed = random_seed
        self.sampler = None
        self.original_distribution = None
        self.resampled_distribution = None
        self.rng = make_rng(random_seed)
        # CHANGED (F2): instance attribute, not module CONFIG.
        self.ablation_mode = ablation_mode

    def get_sampler(self):
        if self.technique == 'SMOTE':
            return SMOTE(random_state=self.random_seed)
        if self.technique == 'ADASYN':
            return ADASYN(random_state=self.random_seed)
        if self.technique == 'BorderlineSMOTE':
            return BorderlineSMOTE(random_state=self.random_seed)
        if self.technique == 'SMOTEENN':
            return SMOTEENN(random_state=self.random_seed)
        if self.technique == 'SMOTETomek':
            return SMOTETomek(random_state=self.random_seed)
        return None

    def fit_resample(self, X, y):
        print(f"📊 Applying {self.technique} for class imbalance...")
        # CHANGED (F2): instance attribute.
        if self.ablation_mode == 'no_smote':
            print("   ⚠️ ABLATION: SMOTE disabled")
            return X, y

        self.original_distribution = Counter(y)
        sampler = self.get_sampler()
        if sampler is None:
            return X, y

        min_c = min(self.original_distribution.values())
        max_c = max(self.original_distribution.values())
        if min_c < 2 or (max_c / min_c) < 2:
            print("   ℹ️ Imbalance ratio < 2 or min class < 2, skipping")
            return X, y

        try:
            if len(X) > 200000:
                print(f"   ⚠️ Large dataset ({len(X):,}), limiting SMOTE")
                sample_size = min(100000, len(X))
                idx = self.rng.choice(len(X), sample_size, replace=False)
                X_sub = X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]
                y_sub = y.iloc[idx] if isinstance(y, pd.Series) else y[idx]
                X_res, y_res = sampler.fit_resample(X_sub, y_sub)
                remaining = np.setdiff1d(np.arange(len(X)), idx)
                X_rest = X.iloc[remaining] if isinstance(X, pd.DataFrame) else X[remaining]
                y_rest = y.iloc[remaining] if isinstance(y, pd.Series) else y[remaining]
                X_res = np.vstack([X_res, np.asarray(X_rest)])
                y_res = np.concatenate([y_res, np.asarray(y_rest)])
            else:
                X_res, y_res = sampler.fit_resample(X, y)

            self.resampled_distribution = Counter(y_res)
            print(f"   Original: {dict(self.original_distribution)}")
            print(f"   Resampled: {dict(self.resampled_distribution)}")
            return X_res, y_res
        except Exception as e:
            print(f"   ⚠️ Resampling failed: {e}")
            traceback.print_exc()
            return X, y

# ============================================================================
# SOTA BASELINE MODELS
# ============================================================================

class SOTABaselines:
    def __init__(self, random_seed=42):
        self.random_seed = random_seed
        self.results = {}
        self.training_times = {}
        self.inference_times = {}
        self.rng = make_rng(random_seed)

    def _fit_and_eval(self, model, X_train, y_train, X_test, y_test, name):
        t0 = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - t0
        t1 = time.time()
        y_pred = model.predict(X_test)
        infer_time = time.time() - t1
        self.training_times[name] = train_time
        self.inference_times[name] = infer_time
        return self._compute_metrics(y_test, y_pred, model, X_test)

    def train_xgboost(self, X_train, y_train, X_test, y_test):
        print("   Training XGBoost...")
        if not XGB_AVAILABLE:
            return {'error': 'XGBoost not available', 'status': 'FAILED'}
        try:
            subsample = 0.8 if len(X_train) > 200000 else 1.0
            model = xgb.XGBClassifier(
                n_estimators=200, max_depth=8, learning_rate=0.08,
                random_state=self.random_seed, use_label_encoder=False,
                eval_metric='logloss', n_jobs=-1, subsample=subsample,
                colsample_bytree=0.8, reg_lambda=1.0, min_child_weight=2
            )
            return self._fit_and_eval(model, X_train, y_train, X_test, y_test,
                                      'xgboost')
        except Exception as e:
            tb = traceback.format_exc()
            print(f"   ❌ XGBoost FAILED: {e}")
            print(tb)
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    def train_naive_bayes(self, X_train, y_train, X_test, y_test):
        print("   Training Naive Bayes...")
        try:
            return self._fit_and_eval(GaussianNB(), X_train, y_train,
                                      X_test, y_test, 'naive_bayes')
        except Exception as e:
            tb = traceback.format_exc()
            print(f"   ❌ Naive Bayes FAILED: {e}\n{tb}")
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    def _subsample_dl(self, X, y, cap):
        if len(X) > cap:
            idx = self.rng.choice(len(X), cap, replace=False)
            return X[idx], y[idx]
        return X, y

    def train_dnn(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training DNN...")
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available', 'status': 'FAILED'}
        try:
            Xs, ys = self._subsample_dl(X_train, y_train, 100000)
            model = keras.Sequential([
                layers.Input(shape=(X_train.shape[1],)),
                layers.Dense(64, activation='relu',
                             kernel_regularizer=keras.regularizers.l2(0.001)),
                layers.Dropout(0.3),
                layers.Dense(32, activation='relu',
                             kernel_regularizer=keras.regularizers.l2(0.001)),
                layers.Dropout(0.3),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(optimizer=keras.optimizers.Adam(0.001),
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
            es = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            t0 = time.time()
            model.fit(Xs, ys, validation_split=0.2,
                      epochs=CONFIG['deep_learning_epochs'],
                      batch_size=CONFIG['deep_learning_batch_size'],
                      callbacks=[es], verbose=0)
            self.training_times['dnn'] = time.time() - t0
            t1 = time.time()
            y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
            self.inference_times['dnn'] = time.time() - t1
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            tb = traceback.format_exc()
            print(f"   ❌ DNN FAILED: {e}\n{tb}")
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    def train_lstm(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training LSTM...")
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available', 'status': 'FAILED'}
        try:
            Xs, ys = self._subsample_dl(X_train, y_train, 50000)
            Xtr = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)
            Xte = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
            model = keras.Sequential([
                layers.LSTM(32, return_sequences=True,
                            input_shape=(X_train.shape[1], 1)),
                layers.Dropout(0.3),
                layers.LSTM(16),
                layers.Dropout(0.3),
                layers.Dense(8, activation='relu'),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(optimizer=keras.optimizers.Adam(0.001),
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
            es = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            t0 = time.time()
            model.fit(Xtr, ys, validation_split=0.2, epochs=10,
                      batch_size=CONFIG['deep_learning_batch_size'],
                      callbacks=[es], verbose=0)
            self.training_times['lstm'] = time.time() - t0
            t1 = time.time()
            y_pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
            self.inference_times['lstm'] = time.time() - t1
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            tb = traceback.format_exc()
            print(f"   ❌ LSTM FAILED: {e}\n{tb}")
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    def train_cnn_lstm(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training CNN-LSTM...")
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available', 'status': 'FAILED'}
        try:
            Xs, ys = self._subsample_dl(X_train, y_train, 50000)
            Xtr = Xs.reshape(Xs.shape[0], Xs.shape[1], 1)
            Xte = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
            model = keras.Sequential([
                layers.Conv1D(filters=16, kernel_size=3, activation='relu',
                              input_shape=(X_train.shape[1], 1), padding='same'),
                layers.MaxPooling1D(pool_size=2),
                layers.LSTM(16, return_sequences=True),
                layers.LSTM(8),
                layers.Dropout(0.3),
                layers.Dense(8, activation='relu'),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(optimizer=keras.optimizers.Adam(0.001),
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])
            es = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            t0 = time.time()
            model.fit(Xtr, ys, validation_split=0.2, epochs=10,
                      batch_size=CONFIG['deep_learning_batch_size'],
                      callbacks=[es], verbose=0)
            self.training_times['cnn_lstm'] = time.time() - t0
            t1 = time.time()
            y_pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
            self.inference_times['cnn_lstm'] = time.time() - t1
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            tb = traceback.format_exc()
            print(f"   ❌ CNN-LSTM FAILED: {e}\n{tb}")
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    def _compute_metrics(self, y_true, y_pred, model, X_test):
        try:
            return {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision_macro': precision_score(y_true, y_pred,
                                                   average='macro', zero_division=0),
                'recall_macro': recall_score(y_true, y_pred,
                                             average='macro', zero_division=0),
                'f1_macro': f1_score(y_true, y_pred,
                                     average='macro', zero_division=0),
                'precision_weighted': precision_score(y_true, y_pred,
                                                      average='weighted', zero_division=0),
                'recall_weighted': recall_score(y_true, y_pred,
                                                average='weighted', zero_division=0),
                'f1_weighted': f1_score(y_true, y_pred,
                                        average='weighted', zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
                'n_samples': int(len(y_true)),
                'status': 'OK',
                'classification_report': classification_report(
                    y_true, y_pred, zero_division=0),
                'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
                'per_class_errors': self._per_class_errors(y_true, y_pred),
            }
        except Exception as e:
            tb = traceback.format_exc()
            return {'error': str(e), 'traceback': tb, 'status': 'FAILED'}

    @staticmethod
    def _per_class_errors(y_true, y_pred):
        out = {}
        for cls in np.unique(y_true):
            mask = (y_true == cls)
            total = int(mask.sum())
            errs = int(((y_true == cls) & (y_true != y_pred)).sum())
            out[int(cls)] = {
                'total': total,
                'errors': errs,
                'error_rate': (errs / total) if total else 0.0,
            }
        return out

    def run_all_baselines(self, X_train, y_train, X_test, y_test):
        print("\n" + "=" * 60)
        print("RUNNING SOTA BASELINE COMPARISONS")
        print("=" * 60)
        n_classes = len(np.unique(y_train))
        results = {}
        if XGB_AVAILABLE:
            results['xgboost'] = self.train_xgboost(X_train, y_train,
                                                    X_test, y_test)
        else:
            results['xgboost'] = {'error': 'XGBoost not installed',
                                  'status': 'FAILED'}
        results['naive_bayes'] = self.train_naive_bayes(X_train, y_train,
                                                        X_test, y_test)
        if TF_AVAILABLE and n_classes > 1:
            results['dnn'] = self.train_dnn(X_train, y_train,
                                            X_test, y_test, n_classes)
            if X_train.shape[1] > 5:
                results['lstm'] = self.train_lstm(X_train, y_train,
                                                  X_test, y_test, n_classes)
                results['cnn_lstm'] = self.train_cnn_lstm(X_train, y_train,
                                                          X_test, y_test, n_classes)
        self.results = results
        return results

# ============================================================================
# ENHANCED TWO-STEP FRAMEWORK
# ============================================================================

class EnhancedTwoStepFramework:
    def __init__(self, model_type='random_forest', use_smote=True,
                 smote_type='SMOTE', random_seed=42, ablation_mode=None):
        self.model_type = model_type
        self.use_smote = use_smote
        self.smote_type = smote_type
        self.random_seed = random_seed
        self.filter_model = None
        self.ovr_models = {}
        self.classes = None
        self.class_distribution = None
        self.training_time = 0
        self.inference_time = 0
        self.feature_importances = None
        self.error_analysis = {}
        # CHANGED (F2): ablation_mode threaded through to the handler.
        self.ablation_mode = ablation_mode
        self.imbalance_handler = AdvancedImbalanceHandler(
            technique=smote_type, random_seed=random_seed,
            ablation_mode=ablation_mode
        )
        self.rng = make_rng(random_seed)

    def _create_model(self, class_weight=None):
        return RandomForestClassifier(
            n_estimators=200, max_depth=16, min_samples_split=4,
            min_samples_leaf=1, max_features='sqrt',
            class_weight=class_weight or 'balanced_subsample',
            random_state=self.random_seed, n_jobs=-1
        )

    def _create_calibrated_model(self, class_weight=None):
        base = self._create_model(class_weight)
        return CalibratedClassifierCV(base, method='sigmoid', cv=3)

    def train_phase1_filter(self, X, y):
        print("🔒 Phase 1: Training Binary Filter...")
        t0 = time.time()
        y_binary = (y != 0).astype(int)
        self.class_distribution = {
            'normal': int((y_binary == 0).sum()),
            'anomaly': int((y_binary == 1).sum())
        }
        print(f"   Normal: {self.class_distribution['normal']:,}, "
              f"Anomaly: {self.class_distribution['anomaly']:,}")

        if self.class_distribution['anomaly'] == 0:
            class DummyModel:
                def predict(self, X): return np.zeros(len(X))
                def predict_proba(self, X): return np.ones((len(X), 2))
                def fit(self, X, y): pass
            self.filter_model = DummyModel()
            return self.filter_model

        if self.use_smote:
            X_bal, y_bal = self.imbalance_handler.fit_resample(X, y_binary)
        else:
            X_bal, y_bal = X, y_binary

        self.filter_model = self._create_calibrated_model(class_weight='balanced')
        self.filter_model.fit(X_bal, y_bal)
        print(f"   Filter training took {time.time()-t0:.2f}s")
        return self.filter_model

    def train_phase2_ovr_models(self, X, y):
        print("🎯 Phase 2: Training OvR Specialized Models...")
        t0 = time.time()

        # CHANGED (F2): ablation now actually takes effect.
        if self.ablation_mode == 'no_cascade':
            print("   ⚠️ ABLATION: cascade disabled — training single multi-class RF")
            anomaly_mask = (y != 0)
            if isinstance(X, pd.DataFrame):
                X_anom = X.loc[anomaly_mask]
                y_anom = y.loc[anomaly_mask]
            else:
                X_anom = X[anomaly_mask]
                y_anom = y[anomaly_mask]
            self.classes = sorted(np.unique(y_anom))
            flat_model = self._create_model(class_weight='balanced')
            flat_model.fit(X_anom, y_anom)
            self.ovr_models = {'__flat__': flat_model}
            print(f"   ✅ Flat model trained on {len(y_anom):,} anomaly rows")
            self.training_time = time.time() - t0
            return self.ovr_models

        self.classes = sorted(np.unique(y[y != 0]))
        print(f"   Attack classes: {len(self.classes)}")
        if len(self.classes) == 0:
            return self.ovr_models

        for attack_class in tqdm(self.classes, desc="OvR models"):
            y_ovr = (y == attack_class).astype(int)
            class_count = int(y_ovr.sum())
            total_count = len(y_ovr)
            if class_count > 0:
                if (self.use_smote and class_count < total_count / 3
                        and class_count >= 2):
                    try:
                        X_bal, y_bal = self.imbalance_handler.fit_resample(X, y_ovr)
                    except Exception:
                        X_bal, y_bal = X, y_ovr
                else:
                    X_bal, y_bal = X, y_ovr
                model = self._create_calibrated_model(class_weight='balanced')
                model.fit(X_bal, y_bal)
                self.ovr_models[attack_class] = model
                print(f"   Class {attack_class}: {class_count:,} samples, "
                      f"IR={total_count/max(class_count,1):.1f}")
                if self.use_smote and class_count < total_count / 3:
                    try:
                        del X_bal, y_bal
                    except Exception:
                        pass
                    cleanup_memory()
        self.training_time = time.time() - t0
        print(f"   ✅ OvR training took {self.training_time:.2f}s")
        return self.ovr_models

    def fit(self, X, y):
        print("🚀 Training Enhanced Two-Step Framework...")
        total_start = time.time()
        self.train_phase1_filter(X, y)
        self.train_phase2_ovr_models(X, y)
        if hasattr(self.filter_model, 'feature_importances_'):
            self.feature_importances = self.filter_model.feature_importances_
        print(f"✅ Total training time: {time.time()-total_start:.2f}s")
        return self

    def predict(self, X):
        t0 = time.time()
        y_binary_pred = self.filter_model.predict(X)
        y_pred = np.zeros(len(X), dtype=int)

        if '__flat__' in self.ovr_models:
            anomaly_indices = np.where(y_binary_pred == 1)[0]
            if len(anomaly_indices) > 0:
                X_anom = (X.iloc[anomaly_indices]
                          if isinstance(X, pd.DataFrame)
                          else X[anomaly_indices])
                flat_pred = self.ovr_models['__flat__'].predict(X_anom)
                y_pred[anomaly_indices] = flat_pred
            self.inference_time = time.time() - t0
            return y_pred

        anomaly_indices = np.where(y_binary_pred == 1)[0]
        if len(anomaly_indices) > 0 and len(self.ovr_models) > 0:
            X_anom = (X.iloc[anomaly_indices]
                      if isinstance(X, pd.DataFrame)
                      else X[anomaly_indices])
            prob_matrix = np.zeros((len(anomaly_indices), len(self.ovr_models)))
            classes_list = list(self.ovr_models.keys())
            for j, cls in enumerate(classes_list):
                try:
                    p = self.ovr_models[cls].predict_proba(X_anom)
                    prob_matrix[:, j] = p[:, 1] if p.shape[1] > 1 else p.ravel()
                except Exception:
                    prob_matrix[:, j] = self.ovr_models[cls].predict(X_anom)
            row_sums = prob_matrix.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1.0
            prob_norm = prob_matrix / row_sums
            winner_idx = np.argmax(prob_norm, axis=1)
            for k, orig in enumerate(anomaly_indices):
                y_pred[orig] = classes_list[winner_idx[k]]
        self.inference_time = time.time() - t0
        return y_pred

    def analyze_errors(self, X_test, y_test, y_pred):
        print("\n" + "=" * 60)
        print("🔍 DEEP ERROR ANALYSIS")
        print("=" * 60)
        errors = {}
        error_indices = np.where(y_test != y_pred)[0]
        if len(error_indices) == 0:
            print("   ✅ No errors found!")
            return errors
        print(f"   Total errors: {len(error_indices)}/{len(y_test)} "
              f"({len(error_indices)/len(y_test)*100:.2f}%)")
        for class_label in np.unique(y_test):
            class_mask = (y_test == class_label)
            class_errors = int(((y_test == class_label) & (y_test != y_pred)).sum())
            class_total = int(class_mask.sum())
            err_rate = class_errors / class_total if class_total else 0.0
            errors[f'class_{int(class_label)}'] = {
                'total': class_total, 'errors': class_errors,
                'error_rate': err_rate
            }
            print(f"   Class {class_label}: {class_errors}/{class_total} "
                  f"({err_rate*100:.2f}%)")
        cm = confusion_matrix(y_test, y_pred)
        confusions = [(int(i), int(j), int(cm[i][j]))
                      for i in range(len(cm)) for j in range(len(cm))
                      if i != j and cm[i][j] > 0]
        confusions.sort(key=lambda x: x[2], reverse=True)
        errors['confusion_patterns'] = confusions[:5]
        self.error_analysis = errors
        return errors

# ============================================================================
# SHAP EXPLAINER  (F1 — robust to list vs 3-D array)
# ============================================================================

def run_shap_analysis(framework, X_train, X_test, feature_names,
                      n_samples=100, seed=42, top_k=10):
    """
    CHANGED (F1): robust to shap returning a list OR a 3-D array.
    Coerces mean|SHAP| to a plain 1-D float array before sorting so tuple
    comparison is always well-defined.
    """
    print("\n🔍 Running SHAP analysis on Phase-1 filter...")
    try:
        rng = make_rng(seed)
        n = min(n_samples, len(X_test))
        idx = rng.choice(len(X_test), n, replace=False)
        X_sample = (X_test.iloc[idx] if isinstance(X_test, pd.DataFrame)
                    else X_test[idx])

        # Unwrap CalibratedClassifierCV (sklearn >= 1.2 vs older)
        base = framework.filter_model
        if hasattr(base, 'calibrated_classifiers_'):
            cc = base.calibrated_classifiers_[0]
            base = getattr(cc, 'estimator', None) or getattr(cc, 'base_estimator', None)

        explainer = shap.TreeExplainer(base)
        shap_values = explainer.shap_values(X_sample, check_additivity=False)

        # --- Robust extraction of class-1 shap values -------------------
        if isinstance(shap_values, list):
            sv = shap_values[1] if len(shap_values) > 1 else shap_values[0]
        else:
            sv = np.asarray(shap_values)
            if sv.ndim == 3:
                # (n_samples, n_features, n_classes) -> pick anomaly class
                sv = sv[..., -1]
            elif sv.ndim == 1:
                sv = sv.reshape(1, -1)
        sv = np.asarray(sv)
        if sv.ndim == 1:
            sv = sv.reshape(1, -1)

        # --- Reduce to a 1-D per-feature importance ---------------------
        mean_abs = np.abs(sv).mean(axis=0)
        mean_abs = np.asarray(mean_abs).ravel()

        # If the shape still does not match n_features, realign.
        if mean_abs.shape[0] != len(feature_names):
            arr = np.abs(sv)
            if arr.ndim == 2:
                if arr.shape[1] == len(feature_names):
                    mean_abs = arr.mean(axis=0)
                elif arr.shape[0] == len(feature_names):
                    mean_abs = arr.mean(axis=1)
            k = min(mean_abs.shape[0], len(feature_names))
            mean_abs = mean_abs[:k]
            feature_names = list(feature_names)[:k]

        # Force scalars before sorting so tuple comparison is well-defined.
        pairs = [(str(f), float(v))
                 for f, v in zip(feature_names, mean_abs.tolist())]
        ranked = sorted(pairs, key=lambda x: x[1], reverse=True)
        top = [{'feature': f, 'mean_abs_shap': v} for f, v in ranked[:top_k]]
        print(f"   Top-{top_k} features by mean |SHAP|: "
              f"{[t['feature'] for t in top]}")
        return {'top_features': top, 'n_explained': n}
    except Exception as e:
        tb = traceback.format_exc()
        print(f"   ⚠️ SHAP failed: {e}\n{tb}")
        return {'error': str(e), 'traceback': tb}

# ============================================================================
# ENHANCED PIPELINE
# ============================================================================

class EnhancedNIDSPipeline:
    def __init__(self, config):
        self.config = config
        self.dataset_config = config.get('dataset_config', {})
        self.seed = config.get('random_seed', 42)
        self.rng = make_rng(self.seed)
        # CHANGED (F2): ablation_mode passed into both preprocessor and
        # framework so every stage sees its own run's setting.
        self.preprocessor = MemoryOptimizedDataPreprocessor(
            dataset_name=config.get('dataset_name', 'unknown'),
            leakage_threshold=config.get('leakage_mi_threshold', 0.95),
            seed=self.seed,
            ablation_mode=config.get('ablation_mode'),
        )
        self.framework = EnhancedTwoStepFramework(
            model_type=config.get('model_type', 'random_forest'),
            use_smote=config.get('use_smote', True),
            smote_type=config.get('smote_type', 'SMOTE'),
            random_seed=self.seed,
            ablation_mode=config.get('ablation_mode'),
        )
        self.sota_baselines = SOTABaselines(random_seed=self.seed)
        self.results = {}
        self.total_time = 0

    def run_pipeline(self, df=None, label_col='Label'):
        print("=" * 80)
        print("🚀 ENHANCED AI-NIDS PIPELINE  (v3.1)")
        print(f"📊 Dataset: {self.config.get('dataset_name', 'unknown')}")
        print(f"🧪 Ablation: {self.config.get('ablation_mode', None) or 'none'}")
        print(f"🎲 Seed: {self.seed}")
        print(f"💾 Memory: {get_memory_usage():.2f} GB")
        print("=" * 80)
        pipeline_start = time.time()

        if df is None:
            df = self.preprocessor.load_data_from_kaggle(self.dataset_config)
            if df is None:
                return self.results

        print("\n" + "-" * 60)
        print("STAGE 1: Data Preprocessing & Leakage Prevention")
        print("-" * 60)
        X, y = self.preprocessor.preprocess(df, label_col)
        self.results['leakage_report'] = self.preprocessor.get_leakage_report()
        if len(X) == 0:
            return self.results

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=self.config.get('test_size', 0.2),
                random_state=self.seed, stratify=y)
        except Exception:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=self.config.get('test_size', 0.2),
                random_state=self.seed)
        print(f"   Train: {len(X_train):,}, Test: {len(X_test):,}, "
              f"Classes: {len(np.unique(y_train))}")

        print("\n" + "-" * 60)
        print("STAGE 2: Feature Selection")
        print("-" * 60)

        if self.config.get('ablation_mode') == 'no_feature_selection':
            print("   ⚠️ ABLATION: feature selection disabled")
            selected_features = X_train.columns.tolist()
            X_train_selected = X_train
            X_test_selected = X_test
        else:
            n_features = min(self.config.get('n_features', 30), X_train.shape[1])
            selector = SelectKBest(mutual_info_classif, k=n_features)
            Xtr = selector.fit_transform(X_train, y_train)
            Xte = selector.transform(X_test)
            selected_features = X_train.columns[selector.get_support()].tolist()
            X_train_selected = pd.DataFrame(Xtr, columns=selected_features)
            X_test_selected = pd.DataFrame(Xte, columns=selected_features)

        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train_selected), columns=selected_features)
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test_selected), columns=selected_features)
        self.results['selected_features'] = selected_features
        self.results['feature_count'] = len(selected_features)
        self.results['original_feature_count'] = X.shape[1]
        del X_train, X_test, X_train_selected, X_test_selected
        cleanup_memory()

        if len(X_train_scaled) > 100000:
            print(f"   ⚠️ Large train set ({len(X_train_scaled):,}), "
                  f"stratified common sample")
            X_train_common, y_train_common = stratified_subsample(
                X_train_scaled, y_train, 50000, seed=self.seed)
            X_test_common, y_test_common = stratified_subsample(
                X_test_scaled, y_test, 20000, seed=self.seed)
        else:
            X_train_common, y_train_common = X_train_scaled, y_train
            X_test_common, y_test_common = X_test_scaled, y_test

        print("\n" + "-" * 60)
        print("STAGE 3: Two-Step Framework")
        print("-" * 60)
        self.framework.fit(X_train_common, y_train_common)
        y_pred = self.framework.predict(X_test_common)
        framework_metrics = self._compute_metrics(y_test_common, y_pred)
        framework_metrics['training_time'] = self.framework.training_time
        framework_metrics['inference_time'] = self.framework.inference_time
        self.results['framework_metrics'] = framework_metrics

        self.results['error_analysis'] = self.framework.analyze_errors(
            X_test_common, y_test_common, y_pred)

        if self.config.get('run_shap', False):
            self.results['shap'] = run_shap_analysis(
                self.framework, X_train_common, X_test_common,
                selected_features,
                n_samples=self.config.get('shap_samples', 100),
                seed=self.seed)

        print("\n" + "-" * 60)
        print("STAGE 4: SOTA Baseline Comparisons")
        print("-" * 60)
        sota_results = self.sota_baselines.run_all_baselines(
            X_train_common.values, y_train_common.values,
            X_test_common.values, y_test_common.values)
        self.results['sota_baselines'] = sota_results
        self.results['training_times'] = self.sota_baselines.training_times
        self.results['inference_times'] = self.sota_baselines.inference_times
        self.results['comparison'] = self._compile_comparison(
            framework_metrics, sota_results)

        self.total_time = time.time() - pipeline_start
        self.results['total_time'] = self.total_time
        print("\n" + "=" * 80)
        print("✅ PIPELINE COMPLETE")
        print("=" * 80)
        print(f"   Framework Macro-F1: {framework_metrics.get('f1_macro', 0):.4f}")
        return self.results

    def _compute_metrics(self, y_true, y_pred):
        try:
            return {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision_macro': precision_score(y_true, y_pred,
                                                   average='macro', zero_division=0),
                'recall_macro': recall_score(y_true, y_pred,
                                             average='macro', zero_division=0),
                'f1_macro': f1_score(y_true, y_pred,
                                     average='macro', zero_division=0),
                'precision_weighted': precision_score(y_true, y_pred,
                                                      average='weighted', zero_division=0),
                'recall_weighted': recall_score(y_true, y_pred,
                                                average='weighted', zero_division=0),
                'f1_weighted': f1_score(y_true, y_pred,
                                        average='weighted', zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
                'n_samples': int(len(y_true)),
            }
        except Exception as e:
            return {'error': str(e), 'traceback': traceback.format_exc()}

    def _compile_comparison(self, framework_metrics, sota_results):
        comparison = {'framework': {
            'f1_macro': framework_metrics.get('f1_macro', 0),
            'accuracy': framework_metrics.get('accuracy', 0)
        }}
        for name, results in sota_results.items():
            if 'error' not in results and results.get('status') != 'FAILED':
                comparison[name] = {
                    'f1_macro': results.get('f1_macro', 0),
                    'accuracy': results.get('accuracy', 0)
                }
        return comparison

    def run_cv(self, df, label_col='Label', n_folds=3):
        """
        Stratified k-fold CV on the framework only, returning mean±std
        macro-F1 across folds. Replaces the "incidental Kaggle version
        history" that the paper currently cites for its reliability claim.
        """
        X, y = self.preprocessor.preprocess(df, label_col)
        if len(X) == 0:
            return None
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True,
                              random_state=self.seed)
        fold_scores = []
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            print(f"\n   ── CV fold {fold+1}/{n_folds} ──")
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            ytr, yte = y.iloc[tr], y.iloc[te]
            n_features = min(self.config.get('n_features', 30), Xtr.shape[1])
            sel = SelectKBest(mutual_info_classif, k=n_features).fit(Xtr, ytr)
            Xtr_s = sel.transform(Xtr)
            Xte_s = sel.transform(Xte)
            sc = StandardScaler().fit(Xtr_s)
            Xtr_s, Xte_s = sc.transform(Xtr_s), sc.transform(Xte_s)
            Xtr_s, ytr_s = stratified_subsample(
                pd.DataFrame(Xtr_s), ytr.reset_index(drop=True),
                50000, seed=self.seed + fold)
            fw = EnhancedTwoStepFramework(
                model_type=self.config.get('model_type', 'random_forest'),
                use_smote=self.config.get('use_smote', True),
                smote_type=self.config.get('smote_type', 'SMOTE'),
                random_seed=self.seed + fold,
                ablation_mode=self.config.get('ablation_mode'),
            )
            fw.fit(Xtr_s, ytr_s)
            yp = fw.predict(pd.DataFrame(Xte_s))
            score = f1_score(yte, yp, average='macro', zero_division=0)
            fold_scores.append(score)
            print(f"   Fold {fold+1} macro-F1: {score:.4f}")
        mean, std = float(np.mean(fold_scores)), float(np.std(fold_scores))
        print(f"\n   ✅ CV macro-F1: {mean:.4f} ± {std:.4f}  (n={n_folds})")
        return {'mean_f1_macro': mean, 'std_f1_macro': std, 'folds': fold_scores}

    def export_results(self, output_path):
        print(f"\n📊 Exporting results to {output_path}...")
        json_path = output_path.replace('.csv', '.json')
        with open(json_path, 'w') as f:
            json.dump(self.results, f, default=str, indent=2)

        dataset_name = self.config.get('dataset_name', 'unknown')
        ablation = self.config.get('ablation_mode', None) or 'none'
        seed = self.seed
        ts = datetime.now().isoformat()

        rows = []

        def add(model, metric, value):
            rows.append({
                'dataset': dataset_name,
                'ablation': ablation,
                'seed': seed,
                'model_type': model,
                'metric': metric,
                'value': value,
                'timestamp': ts,
            })

        for metric, value in self.results.get('framework_metrics', {}).items():
            if isinstance(value, (int, float)):
                add('framework', metric, value)

        for model_name, metrics in self.results.get('sota_baselines', {}).items():
            if metrics.get('status') == 'FAILED':
                add(model_name, 'status_failed', 1)
                continue
            for metric, value in metrics.items():
                if isinstance(value, (int, float)):
                    add(model_name, metric, value)

        for model_name, metrics in self.results.get('comparison', {}).items():
            for metric, value in metrics.items():
                if isinstance(value, (int, float)):
                    add(f'comparison_{model_name}', metric, value)

        leakage = self.results.get('leakage_report', {})
        add('leakage', 'leaky_features_count', leakage.get('leaky_count', 0))
        add('leakage', 'preprocessing_time', leakage.get('preprocessing_time', 0))

        for name, t in self.results.get('training_times', {}).items():
            add(name, 'training_time_sec', t)
        for name, t in self.results.get('inference_times', {}).items():
            add(name, 'inference_time_sec', t)

        add('meta', 'n_features_selected', self.results.get('feature_count', 0))
        add('meta', 'n_features_original',
            self.results.get('original_feature_count', 0))

        df_results = pd.DataFrame(rows)
        df_results.to_csv(output_path, index=False)
        print(f"   ✅ Exported {len(df_results)} rows")
        return df_results

# ============================================================================
# PER-DATASET ENTRY POINT
# ============================================================================

def run_enhanced_pipeline_for_dataset(dataset_name, dataset_config,
                                      ablation_mode=None, seed=42):
    print(f"\n{'#'*80}")
    print(f"# DATASET: {dataset_name}   |   ablation={ablation_mode}   |   seed={seed}")
    print(f"{'#'*80}")
    config = CONFIG.copy()
    config['dataset_name'] = dataset_name
    config['dataset_config'] = dataset_config
    config['ablation_mode'] = ablation_mode
    config['random_seed'] = seed
    os.makedirs(config['output_dir'], exist_ok=True)
    pipeline = EnhancedNIDSPipeline(config)
    results = pipeline.run_pipeline()
    tag = f"{dataset_name}_abl-{ablation_mode or 'none'}_seed-{seed}"
    output_path = os.path.join(config['output_dir'],
                               f'enhanced_nids_results_{tag}.csv')
    pipeline.export_results(output_path)
    cleanup_memory()
    return pipeline, results

# ============================================================================
# MAIN  (F5 — diagnostics + failed_runs.csv)
# ============================================================================

def main():
    print("=" * 80)
    print("MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION (v3.1)")
    print("=" * 80)

    # --- Dataset availability with explicit diagnostics ---------------
    present, missing = [], []
    for name, cfg in DATASETS.items():
        if os.path.exists(cfg['path']):
            present.append(name)
            print(f"   ✅ {name}: {cfg['path']}")
        else:
            missing.append(name)
            print(f"   ❌ {name}: {cfg['path']}  (NOT FOUND)")

    print(f"\n✅ Available: {present}")
    print(f"❌ Missing:  {missing}")
    if missing:
        print("\n⚠️ The following datasets will be SKIPPED because their paths")
        print("   do not exist in this environment. Update the 'path' entry in")
        print("   DATASETS for each of these to point at your Kaggle input:")
        for name in missing:
            print(f"     - {name}")
        for guess in ['/kaggle/input', '/kaggle/input/datasets']:
            if os.path.exists(guess):
                print(f"\n   ℹ️ Actual entries under {guess}:")
                try:
                    for entry in sorted(os.listdir(guess))[:30]:
                        print(f"      - {entry}")
                except Exception:
                    pass

    ablation_modes = [None, 'no_leakage', 'no_feature_selection',
                      'no_smote', 'no_cascade']

    all_rows, failed_runs = [], []
    for dataset_name in present:
        for ablation in ablation_modes:
            for seed in CONFIG['repeat_seeds']:
                tag = f"{dataset_name}/abl={ablation}/seed={seed}"
                try:
                    print(f"\n>>> {tag}")
                    pipeline, results = run_enhanced_pipeline_for_dataset(
                        dataset_name, DATASETS[dataset_name],
                        ablation_mode=ablation, seed=seed)
                    fm = results.get('framework_metrics', {})
                    all_rows.append({
                        'dataset': dataset_name,
                        'ablation': ablation or 'none',
                        'seed': seed,
                        'framework_f1_macro': fm.get('f1_macro', 0),
                        'framework_accuracy': fm.get('accuracy', 0),
                        'framework_mcc': fm.get('matthews_corrcoef', 0),
                    })
                except Exception as e:
                    tb = traceback.format_exc()
                    print(f"❌ FAILED {tag}: {e}\n{tb}")
                    failed_runs.append({
                        'dataset': dataset_name,
                        'ablation': ablation or 'none',
                        'seed': seed,
                        'error': str(e),
                        'traceback': tb,
                    })
                    cleanup_memory()

    os.makedirs('/kaggle/working/results/', exist_ok=True)

    if all_rows:
        summary = pd.DataFrame(all_rows)
        summary.to_csv('/kaggle/working/results/repeated_runs_summary.csv',
                       index=False)
        agg = summary.groupby(['dataset', 'ablation']).agg(
            f1_mean=('framework_f1_macro', 'mean'),
            f1_std=('framework_f1_macro', 'std'),
            acc_mean=('framework_accuracy', 'mean'),
            mcc_mean=('framework_mcc', 'mean'),
            n_runs=('framework_f1_macro', 'count'),
        ).reset_index()
        agg.to_csv('/kaggle/working/results/repeated_runs_aggregate.csv',
                   index=False)
        print("\n" + "=" * 80)
        print("📈 REPEATED-RUN SUMMARY")
        print("=" * 80)
        print(agg.to_string(index=False))

    if failed_runs:
        pd.DataFrame(failed_runs).to_csv(
            '/kaggle/working/results/failed_runs.csv', index=False)
        print(f"\n⚠️ {len(failed_runs)} run(s) failed — see failed_runs.csv")

    print("\n" + "=" * 80)
    print("✅ EXECUTION COMPLETE — results in /kaggle/working/results/")
    print("=" * 80)

if __name__ == "__main__":
    main()

✅ SHAP version: 0.51.0
MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION (v3.1)
   ✅ NSL-KDD: /kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert
   ✅ UNSW-NB15: /kaggle/input/datasets/likkisamarthreddy/unsw-nb15
   ✅ CIC-IDS2017: /kaggle/input/datasets/bertvankeulen/cicids-2017
   ✅ CSE-CICIDS2018: /kaggle/input/datasets/shrey213/cicids2018
   ✅ CIC-ToN-IoT: /kaggle/input/datasets/wahidulislambayazid/cic-ton-iot

✅ Available: ['NSL-KDD', 'UNSW-NB15', 'CIC-IDS2017', 'CSE-CICIDS2018', 'CIC-ToN-IoT']
❌ Missing:  []

>>> NSL-KDD/abl=None/seed=42

################################################################################
# DATASET: NSL-KDD   |   ablation=None   |   seed=42
################################################################################
🚀 ENHANCED AI-NIDS PIPELINE  (v3.1)
📊 Dataset: NSL-KDD
🧪 Ablation: none
🎲 Seed: 42
💾 Memory: 0.95 GB
📂 Loading NSL-KDD...
   Found 1 files under /kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert
   Selected: NSL_KDD_BERT_Embeddi

OvR models: 100%|██████████| 1/1 [00:05<00:00,  5.88s/it]

   Class 1: 25,034 samples, IR=2.0
   ✅ OvR training took 5.89s
✅ Total training time: 11.62s



🔍 DEEP ERROR ANALYSIS
   Total errors: 170/20000 (0.85%)
   Class 0: 87/9987 (0.87%)
   Class 1: 83/10013 (0.83%)

🔍 Running SHAP analysis on Phase-1 filter...
   Top-10 features by mean |SHAP|: ['dst_bytes', 'service', 'flag_SF', 'logged_in', 'dst_host_rerror_rate', 'diff_srv_rate', 'dst_host_same_srv_rate', 'dst_host_srv_count', 'count', 'dst_host_diff_srv_rate']

------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...


I0000 00:00:1789508790.081101      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789508790.083965      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1789508794.497478     208 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

   Class 1: 26,483 samples, IR=1.9
   ✅ OvR training took 15.05s
✅ Total training time: 30.32s



🔍 DEEP ERROR ANALYSIS
   Total errors: 158/20000 (0.79%)
   Class 0: 150/9407 (1.59%)
   Class 1: 8/10593 (0.08%)

🔍 Running SHAP analysis on Phase-1 filter...
   Top-10 features by mean |SHAP|: ['src_port', 'totlen_fwd_pkts', 'dst_ip', 'fwd_pkt_len_mean', 'init_bwd_win_byts', 'fwd_seg_size_avg', 'init_fwd_win_byts', 'fwd_header_len', 'subflow_fwd_byts', 'bwd_pkt_len_std']

------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...
   Training LSTM...
   Training CNN-LSTM...

✅ PIPELINE COMPLETE
   Framework Macro-F1: 0.9921

📊 Exporting results to /kaggle/working/results/enhanced_nids_results_CIC-ToN-IoT_abl-no_smote_seed-44.csv...
   ✅ Exported 88 rows
   Memory after cleanup: 12.18 GB

>>> CIC-ToN-IoT/abl=no_smote/seed=45

#############################################################

OvR models: 100%|██████████| 1/1 [00:15<00:00, 15.50s/it]

   Class 1: 26,483 samples, IR=1.9
   ✅ OvR training took 15.51s
✅ Total training time: 30.72s



🔍 DEEP ERROR ANALYSIS
   Total errors: 173/20000 (0.86%)
   Class 0: 162/9407 (1.72%)
   Class 1: 11/10593 (0.10%)

🔍 Running SHAP analysis on Phase-1 filter...
   Top-10 features by mean |SHAP|: ['src_port', 'dst_ip', 'init_bwd_win_byts', 'fwd_pkt_len_mean', 'subflow_fwd_byts', 'fwd_header_len', 'totlen_fwd_pkts', 'fwd_seg_size_avg', 'init_fwd_win_byts', 'bwd_pkt_len_std']

------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...
   Training LSTM...
   Training CNN-LSTM...

✅ PIPELINE COMPLETE
   Framework Macro-F1: 0.9913

📊 Exporting results to /kaggle/working/results/enhanced_nids_results_CIC-ToN-IoT_abl-no_smote_seed-45.csv...
   ✅ Exported 88 rows
   Memory after cleanup: 12.24 GB

>>> CIC-ToN-IoT/abl=no_cascade/seed=42

##########################################################